# Data cleaning pipeline

The photographer usually **crops or straightens** a photo while editing, so the edited file is smaller than the original and shows only part of it. The previous version kept a pair only if the edited image was exactly 6720×4480 (or exactly the size of its original), which threw away every cropped edit — most of the data. This version finds where the edited image sits inside the original and cuts the original the same way, so cropped pairs are kept.

1. **Pair** files in `data/raw_data/{original,edited}` by photo ID: `UGSL4055-已增强-降噪.jpg` ↔ `4055.jpg` → `4055`. Raw files are only read — never renamed, moved or overwritten.
2. **Align** each pair: match SIFT features between the edited image and its original and fit a similarity transform (crop offset + straighten angle + scale). Pairs that cannot be aligned, or whose content still differs after alignment (another frame of a burst, a wrong file), are rejected. Kept pairs and their transforms go to `data/matched_pairs.json`.
3. **Write** each kept pair to `data/720*480/{original,edited}`: warp the original onto the edited frame, then grayscale → rotate portrait to landscape → fit 720×480 → pad white, as before.
4. **Alignment QA**: spot-check random pairs, the kept pairs closest to the rejection threshold, and the rejected pairs.

Before running, put **all** raw files back into `raw_data/original` and `raw_data/edited` — including the ones an earlier version moved to `raw_data_rejected/`.

In [ ]:
import os
import re
import json
import numpy as np
import cv2
from PIL import Image, ImageOps

Image.MAX_IMAGE_PIXELS = None   # some edited files are ~98 MP, above PIL's safety limit; these are our own files

# Paths assume this notebook lives in <project>/code/
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data")
RAW_DIR = os.path.join(DATA_DIR, "raw_data")
OUT_DIR = os.path.join(DATA_DIR, "720*480")
JSON_PATH = os.path.join(DATA_DIR, "matched_pairs.json")
FOLDERS = ["original", "edited"]
IMG_EXTS = (".jpg", ".jpeg", ".png")   # matched case-insensitively
TARGET = (720, 480)  # (width, height) — landscape 3:2

# Alignment (Step 2)
ALIGN_SIDE = 1600      # long side (px) the images are reduced to for feature matching
MIN_INLIERS = 20       # fewer consistent feature matches than this -> cannot align
MIN_CROP_AREA = 0.1    # the edit must cover at least 10% of the original's area
MIN_COVERAGE = 0.95    # at least 95% of the edited frame must fall inside the original
MAX_MISMATCH = 0.1     # max share of structured regions that disagree after alignment
EXCLUDE = set()        # pairs to drop by hand after QA, e.g. {"4418.jpg"}

for f in FOLDERS:
    p = os.path.join(RAW_DIR, f)
    print(f, "->", p, "| exists:", os.path.isdir(p))

## Step 1 — Pair original ↔ edited by photo ID

The ID is the digits after `UGSL` if present, otherwise the file name up to the first `-` (so `IMG_5082.jpg` → `IMG_5082`). Pairing by ID instead of renaming means different suffixes (`-已增强-降噪`, `-已增强-降噪-2`), `.JPG` / `.jpeg` extensions, and already-renamed files all pair up, and nothing on disk changes. If an ID has several files in one folder (e.g. a second edit `…-2.jpg`), the shortest name is used and the others are listed.

In [ ]:
def photo_id(filename):
    """ID shared by an original and its edit.
    'UGSL4055-已增强-降噪.jpg' -> '4055', '4055.jpg' -> '4055', 'IMG_5082.jpg' -> 'IMG_5082'
    """
    stem = os.path.splitext(filename)[0]
    m = re.match(r"^UGSL(\d+)", stem, re.IGNORECASE)
    return m.group(1) if m else stem.split("-")[0]


def index_folder(folder):
    """photo ID -> image files with that ID in RAW_DIR/folder, shortest name first."""
    ids = {}
    for f in os.listdir(os.path.join(RAW_DIR, folder)):
        if f.lower().endswith(IMG_EXTS) and not f.startswith("."):   # skip macOS '._' metadata files
            ids.setdefault(photo_id(f), []).append(f)
    return {k: sorted(v, key=lambda f: (len(f), f)) for k, v in ids.items()}

In [ ]:
o_ids, e_ids = index_folder("original"), index_folder("edited")
pairs = {pid: (o_ids[pid][0], e_ids[pid][0]) for pid in sorted(set(o_ids) & set(e_ids))
         if pid + ".jpg" not in EXCLUDE}
only_o = sorted(set(o_ids) - set(e_ids))
only_e = sorted(set(e_ids) - set(o_ids))
extra = {pid: o_ids[pid][1:] + e_ids[pid][1:] for pid in pairs if len(o_ids[pid]) > 1 or len(e_ids[pid]) > 1}

print("original files:", sum(map(len, o_ids.values())), "| edited files:", sum(map(len, e_ids.values())))
print("pairs:", len(pairs), "| excluded by hand:", len(EXCLUDE))
print("original without edit (ignored):", len(only_o), only_o[:10])
print("edit without original (ignored):", len(only_e), only_e[:10])
print("IDs with more than one file (first one used):", len(extra))
for pid, fs in list(extra.items())[:10]:
    print("   ", pid, "uses", pairs[pid], "| not used:", fs)
assert pairs, "no pairs found — copy the raw files into raw_data/original and raw_data/edited first"

## Step 2 — Align each pair (find the crop)

Both images are reduced to `ALIGN_SIDE` px and contrast-equalized (CLAHE), SIFT features are matched, and RANSAC fits a similarity transform that maps edited-image pixels to original-image pixels. The transform is stored at full resolution, so Step 3 can cut the original exactly where the photographer cropped.

A pair is **rejected** if
- too few features match (unrelated images, or almost no texture);
- the fitted crop covers < `MIN_CROP_AREA` of the original, or the edited frame falls outside the original (a nonsense fit);
- after alignment the content still differs (`mismatch` > `MAX_MISMATCH`). `mismatch` is the share of structured regions in the central 70% of the frame where the two images disagree (local correlation < 0.5). Local correlation ignores brightness/contrast edits — including local ones such as brightened faces — but not moved content. On the 396 pairs from earlier runs, real pairs score ≤ 0.04, while `4418.jpg` (another frame of a burst) scores 0.20.

In [ ]:
def load_gray(path, long_side=None):
    """Grayscale with the EXIF orientation applied (cameras store portrait shots sideways plus a
    rotate flag). If long_side is given, downscale so the long side is at most long_side; JPEGs
    are then decoded directly at reduced size, which is much faster for 30 MP files.
    Returns (image, scale) with scale = returned size / full-resolution size."""
    with Image.open(path) as im:
        full_long = max(im.size)
        if long_side and full_long > long_side:
            k = full_long / long_side
            im.draft("L", (int(im.size[0] / k), int(im.size[1] / k)))
        im = ImageOps.exif_transpose(im).convert("L")
    if long_side and max(im.size) > long_side:
        k = long_side / max(im.size)
        im = im.resize((max(1, round(im.size[0] * k)), max(1, round(im.size[1] * k))), Image.LANCZOS)
    return im, max(im.size) / full_long


# SIFT is in the main OpenCV module from 4.4 on; older builds have it in opencv-contrib (xfeatures2d)
SIFT = cv2.SIFT_create(4000) if hasattr(cv2, "SIFT_create") else cv2.xfeatures2d.SIFT_create(4000)
MATCHER = cv2.BFMatcher(cv2.NORM_L2)
CLAHE = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))   # evens out exposure so features match across the edit


def estimate_transform(o, e):
    """Similarity transform (2x3) mapping edited px -> original px, and its number of inlier matches."""
    ko, do = SIFT.detectAndCompute(CLAHE.apply(np.asarray(o)), None)
    ke, de = SIFT.detectAndCompute(CLAHE.apply(np.asarray(e)), None)
    if do is None or de is None:
        return None, 0
    # Lowe's ratio test: keep a match only if it is clearly better than the second-best candidate
    good = [p[0] for p in MATCHER.knnMatch(de, do, k=2) if len(p) == 2 and p[0].distance < 0.75 * p[1].distance]
    if len(good) < MIN_INLIERS:
        return None, len(good)
    src = np.float32([ke[m.queryIdx].pt for m in good])
    dst = np.float32([ko[m.trainIdx].pt for m in good])
    M, inliers = cv2.estimateAffinePartial2D(src, dst, method=cv2.RANSAC, ransacReprojThreshold=3.0,
                                             maxIters=5000, confidence=0.999)
    return M, (0 if M is None else int(inliers.sum()))


def mismatch_score(o_aligned, e, side=360, win=9):
    """Share of structured windows in the central 70% of the frame whose local correlation
    is < 0.5. NaN if the centre has too little structure to judge."""
    k = side / max(e.shape)
    size = (max(1, round(e.shape[1] * k)), max(1, round(e.shape[0] * k)))
    a = cv2.GaussianBlur(cv2.resize(o_aligned, size, interpolation=cv2.INTER_AREA).astype(np.float32), (0, 0), 1.0)
    b = cv2.GaussianBlur(cv2.resize(e, size, interpolation=cv2.INTER_AREA).astype(np.float32), (0, 0), 1.0)
    mean = lambda x: cv2.blur(x, (win, win))
    va, vb = mean(a * a) - mean(a) ** 2, mean(b * b) - mean(b) ** 2
    ncc = (mean(a * b) - mean(a) * mean(b)) / np.sqrt(np.maximum(va * vb, 1e-6))
    h, w = ncc.shape
    centre = (slice(int(0.15 * h), h - int(0.15 * h)), slice(int(0.15 * w), w - int(0.15 * w)))
    structured = (va[centre] > 150) & (vb[centre] > 150)
    if structured.sum() < 50:
        return float("nan")
    return float((ncc[centre][structured] < 0.5).mean())


def align_pair(o_path, e_path):
    """Find where the edited image sits in the original. Returns a dict with the transform
    ('M': edited px -> original px at full resolution; 'scale': original px per edited px),
    QA numbers, and 'reject': None if the pair is usable, else the reason."""
    o, so = load_gray(o_path, ALIGN_SIDE)
    e, se = load_gray(e_path, ALIGN_SIDE)
    M, inliers = estimate_transform(o, e)
    info = {"inliers": inliers, "reject": None}
    if M is None or inliers < MIN_INLIERS:
        info["reject"] = "too few matching features"
        return info
    A, t = M[:, :2], M[:, 2]
    scale = float(np.sqrt(abs(np.linalg.det(A))))
    o_arr, e_arr = np.asarray(o), np.asarray(e)
    footprint = cv2.warpAffine(np.full(o_arr.shape, 255, np.uint8), M, e.size,
                               flags=cv2.INTER_NEAREST | cv2.WARP_INVERSE_MAP)   # where the original covers the edited frame
    o_aligned = cv2.warpAffine(o_arr, M, e.size, flags=cv2.INTER_LINEAR | cv2.WARP_INVERSE_MAP,
                               borderMode=cv2.BORDER_REPLICATE)
    info.update(
        M=np.hstack([A * se / so, t[:, None] / so]).tolist(),   # working resolution -> full resolution
        scale=scale * se / so,
        angle=float(np.degrees(np.arctan2(A[1, 0], A[0, 0]))),
        crop_area=e.size[0] * e.size[1] * scale ** 2 / (o.size[0] * o.size[1]),
        coverage=float((footprint > 0).mean()),
        mismatch=mismatch_score(o_aligned, e_arr),
    )
    if info["crop_area"] < MIN_CROP_AREA:
        info["reject"] = "fitted crop is too small (nonsense fit)"
    elif info["coverage"] < MIN_COVERAGE:
        info["reject"] = "edited frame falls outside the original"
    elif info["mismatch"] > MAX_MISMATCH:   # NaN (cannot judge) passes
        info["reject"] = "content differs after alignment"
    return info

In [ ]:
results = {}
for i, (pid, (of, ef)) in enumerate(pairs.items(), 1):
    r = align_pair(os.path.join(RAW_DIR, "original", of), os.path.join(RAW_DIR, "edited", ef))
    r.update(original=of, edited=ef)
    results[pid + ".jpg"] = r
    if i % 50 == 0 or i == len(pairs):
        print(f"aligned {i}/{len(pairs)}")

kept = {name: r for name, r in results.items() if r["reject"] is None}
rejected = {name: r for name, r in results.items() if r["reject"] is not None}

# Save ONLY the kept pairs (with their transforms) — Step 3 reads this file.
with open(JSON_PATH, "w", encoding="utf-8") as fp:
    json.dump(kept, fp, indent=1, ensure_ascii=False)

print(f"\nkept {len(kept)} / {len(results)} pairs -> {JSON_PATH}")
print("  of which the edit is cropped or straightened:",
      sum(1 for r in kept.values() if r["crop_area"] < 0.98 or abs(r["angle"]) > 0.5))
by_reason = {}
for name, r in rejected.items():
    by_reason.setdefault(r["reject"], []).append(name)
for reason, names in by_reason.items():
    print(f"rejected — {reason}: {len(names)}", names[:10])

## Step 3 — Write aligned pairs at 720×480 (grayscale)

For each pair in `matched_pairs.json`, the original is warped onto the edited frame (so it shows exactly what the edited image shows), then both go through the same resize as before: grayscale, portrait rotated to landscape, scaled **proportionally** to fit inside 720×480, leftover area padded **white** (255). Files already in `data/720*480` with the same name are overwritten; other files there are left alone and listed at the end.

In [ ]:
MID_SIDE = 2 * max(TARGET)   # warp at twice the output size, then resize like before


def render_pair(info, long_side):
    """(original warped onto the edited frame, edited), grayscale, long side at most long_side."""
    o_path = os.path.join(RAW_DIR, "original", info["original"])
    e, se = load_gray(os.path.join(RAW_DIR, "edited", info["edited"]), long_side)
    with Image.open(o_path) as im:
        o_long = max(im.size)
    # load the original at about the same pixel scale as e, so the warp barely has to resize it
    o, so = load_gray(o_path, round(o_long * se / info["scale"]))
    M = np.array(info["M"])
    M = np.hstack([M[:, :2] * so / se, M[:, 2:] * so])   # full resolution -> the loaded sizes
    o_warp = cv2.warpAffine(np.asarray(o), M, e.size, flags=cv2.INTER_CUBIC | cv2.WARP_INVERSE_MAP,
                            borderMode=cv2.BORDER_REPLICATE)
    return Image.fromarray(o_warp), e


def fit_to_target(im):
    """Rotate portrait to landscape, scale proportionally to fit 720x480, pad the rest white (255)."""
    tw, th = TARGET
    if im.height > im.width:
        im = im.transpose(Image.ROTATE_90)       # portrait -> landscape
    scale = min(tw / im.width, th / im.height)   # fit inside, keep aspect ratio
    nw, nh = max(1, round(im.width * scale)), max(1, round(im.height * scale))
    canvas = Image.new("L", (tw, th), 255)       # white background (grayscale)
    canvas.paste(im.resize((nw, nh), Image.LANCZOS), ((tw - nw) // 2, (th - nh) // 2))  # center
    return canvas

In [ ]:
with open(JSON_PATH, encoding="utf-8") as fp:
    kept = json.load(fp)
print("to write:", len(kept), "pairs")

for folder in FOLDERS:
    os.makedirs(os.path.join(OUT_DIR, folder), exist_ok=True)
for i, (name, info) in enumerate(kept.items(), 1):
    o, e = render_pair(info, MID_SIDE)
    fit_to_target(o).save(os.path.join(OUT_DIR, "original", name), quality=95, optimize=True)
    fit_to_target(e).save(os.path.join(OUT_DIR, "edited", name), quality=95, optimize=True)
    if i % 100 == 0 or i == len(kept):
        print(f"wrote {i}/{len(kept)}")

In [ ]:
# VERIFY: outputs are 720x480 single-channel (mode 'L'), and both folders hold the same names.
for folder in FOLDERS:
    sizes, modes = {}, {}
    for f in kept:
        with Image.open(os.path.join(OUT_DIR, folder, f)) as im:
            sizes[im.size] = sizes.get(im.size, 0) + 1
            modes[im.mode] = modes.get(im.mode, 0) + 1
    print(folder, "-> sizes:", sizes, "| modes:", modes)

oo = set(f for f in os.listdir(os.path.join(OUT_DIR, "original")) if f.lower().endswith(".jpg"))
ee = set(f for f in os.listdir(os.path.join(OUT_DIR, "edited")) if f.lower().endswith(".jpg"))
print("original vs edited names match 1:1:", oo == ee)
stale = sorted((oo | ee) - set(kept))
print("files in 720*480 NOT written by this run (left over from older runs):", len(stale), stale[:10])

## Step 4 — Alignment QA

Each row is **original | edited | |difference|**. A kept pair should show the same scene with only tone differences: no doubled edges or shifted outlines in the difference map. If a kept pair looks wrong, add it to `EXCLUDE` and re-run; if a rejected pair looks fine, loosen the threshold named in its reason.

In [ ]:
import random
import matplotlib.pyplot as plt

N = 6          # random kept pairs to inspect
K = 4          # kept pairs with the highest mismatch (closest to the threshold)
MAX_REJ = 8    # rejected pairs to show per group
SEED = 0       # change to see different samples


def show(rows, title, diff=True):
    """rows: (label, original, edited) -> one figure row each: original | edited [| |diff|]."""
    if not rows:
        print(title + ": none")
        return
    fig, axes = plt.subplots(len(rows), 3, figsize=(12, 3 * len(rows)), squeeze=False)
    for r, (label, o, e) in enumerate(rows):
        o, e = np.asarray(o, np.float32), np.asarray(e, np.float32)
        panels = [(o, label, "gray"), (e, "edited", "gray")]
        if diff:
            d = np.abs(o - e)
            panels.append((d, f"|diff|  mean={d.mean():.1f}", "magma"))
        for c in range(3):
            axes[r, c].axis("off")
        for c, (img, t, cmap) in enumerate(panels):
            axes[r, c].imshow(img, cmap=cmap, vmin=0, vmax=255)
            axes[r, c].set_title(t, fontsize=9)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def load_out(name):
    return [Image.open(os.path.join(OUT_DIR, folder, name)).convert("L") for folder in FOLDERS]


random.seed(SEED)
sample = random.sample(sorted(kept), min(N, len(kept)))
show([(f, *load_out(f)) for f in sample], "Random kept pairs")

scored = [f for f in kept if not np.isnan(kept[f]["mismatch"])]
closest = sorted(scored, key=lambda f: -kept[f]["mismatch"])[:K]
show([(f"{f}  mismatch={kept[f]['mismatch']:.3f}", *load_out(f)) for f in closest],
     "Kept pairs closest to the mismatch threshold")

differs = [(f, r) for f, r in rejected.items() if r["reject"] == "content differs after alignment"]
show([(f"{f} — {r['reject']}", *render_pair(r, max(TARGET))) for f, r in differs[:MAX_REJ]],
     "Rejected: content differs after alignment (shown aligned)")
unaligned = [(f, r) for f, r in rejected.items() if r["reject"] != "content differs after alignment"]
show([(f"{f} — {r['reject']}", load_gray(os.path.join(RAW_DIR, "original", r["original"]), max(TARGET))[0],
       load_gray(os.path.join(RAW_DIR, "edited", r["edited"]), max(TARGET))[0]) for f, r in unaligned[:MAX_REJ]],
     "Rejected: no usable alignment (files as they are)", diff=False)